# 4D Gaussian Splatting (D-NeRF)

Train and render dynamic 3D scenes using [4D Gaussian Splatting](https://github.com/Tasmay-Tibrewal/4DGaussians) on all 8 D-NeRF scenes: Bouncing Balls, Hell Warrior, Hook, Jumping Jacks, Lego, Mutant, Standup, and T-Rex.

**Requirements:** GPU with CUDA support, PyTorch with CUDA enabled.

## Setup

Clone the repository, install dependencies, and download the D-NeRF dataset.

In [ ]:
import os

# Detect environment: use /content/ on Colab, otherwise use the current working directory
if os.path.exists('/content') and 'COLAB_GPU' in os.environ:
    BASE_DIR = '/content'
else:
    BASE_DIR = os.getcwd()

REPO_DIR = os.path.join(BASE_DIR, '4DGaussians')
DATA_DIR = os.path.join(BASE_DIR, 'test', 'data')

print(f'BASE_DIR: {BASE_DIR}')
print(f'REPO_DIR: {REPO_DIR}')
print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
%cd {BASE_DIR}
!git clone https://github.com/Tasmay-Tibrewal/4DGaussians
%cd 4DGaussians
!git submodule update --init --recursive

!pip install -r requirements.txt
!pip install -e submodules/depth-diff-gaussian-rasterization
!pip install -e submodules/simple-knn

In [ ]:
!sudo apt-get install libglm-dev

In [ ]:
!pip3 install torch torchvision torchaudio

In [ ]:
import os
test_dir = os.path.join(BASE_DIR, 'test')
os.makedirs(test_dir, exist_ok=True)
%cd {test_dir}
!wget https://huggingface.co/camenduru/4DGaussians/resolve/main/data/data.zip
!unzip data.zip

## Utility

Define `display_video` helper used by all scene sections below.

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os
import glob

def display_video(video_path):
  mp4 = open(video_path,'rb').read()
  data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
  display(HTML("""
  <video width=1000 controls>
    <source src="%s" type="video/mp4">
  </video>
  """ % data_url))

## Train, Render, and Display All Scenes

Loop over all 8 D-NeRF scenes: Bouncing Balls, Hell Warrior, Hook, Jumping Jacks, Lego, Mutant, Standup, and T-Rex.

In [ ]:
import os

scenes = [
    "bouncingballs",
    "hellwarrior",
    "hook",
    "jumpingjacks",
    "lego",
    "mutant",
    "standup",
    "trex",
]

for scene in scenes:
    print(f"\n{'='*60}")
    print(f"  Processing scene: {scene}")
    print(f"{'='*60}\n")

    data_path = os.path.join(DATA_DIR, scene)
    config_path = f"arguments/dnerf/{scene}.py"
    exp_name = f"dnerf/{scene}"
    model_output = f"output/dnerf/{scene}/"
    video_path = os.path.join(REPO_DIR, "output", "dnerf", scene, "video", "ours_20000", "video_rgb.mp4")

    %cd {REPO_DIR}
    !python train.py -s {data_path} --port 6017 --expname "{exp_name}" --configs {config_path}
    !python render.py --model_path "{model_output}" --skip_train --configs {config_path}

    display_video(video_path)